In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

trainData = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
testData = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [ ]:
import torch
from torch import nn , save , load
from torch.optim import Adam
from torch.utils.data import DataLoader , TensorDataset
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
print(trainData.shape)
trainData.head()

In [ ]:
#AS A REMINDER OF HOW SLICING WORKS:
array = [[1,2,3],[4,5,6],[7,8,9]]
array = np.array(array)
print(array[1:2,:2])
arraydf = pd.DataFrame(array)
print(arraydf.iloc[1:2,:2].values)

In [ ]:
features = trainData.iloc[:30000, 1:].values #returns an numpy array
labels = trainData.iloc[:30000, 0].values #returns an numpy array
testF = trainData.iloc[30000:,1:].values
testL = trainData.iloc[30000:,0].values
features = features / 255.0
testF = testF / 225.0
features_tensor = torch.tensor(features, dtype=torch.float32)  
features_tensor = features_tensor.view(-1, 1, 28, 28)  # Reshape to match [batch_size, channels, height, width]
testF_tensor = torch.tensor(testF, dtype=torch.float32)
testF_tensor = testF_tensor.view(-1,1,28,28)
labels_tensor = torch.tensor(labels, dtype=torch.long)
testL_tensor = torch.tensor(testL, dtype=torch.long)

print(testF_tensor.shape)
print(testL_tensor.shape)
tData = TensorDataset(testF_tensor, testL_tensor)
tdataset = DataLoader(tData, batch_size=12000)

# Create a TensorDataset
data = TensorDataset(features_tensor, labels_tensor)
# Create Dataloader
dataset = DataLoader(data, batch_size=32, shuffle=True)

#torch.Size([64, 1, 28, 28])
#torch.Size([64])
# Check shape and image
for batch_features, batch_labels in dataset:
    print("Batch Features Shape:", batch_features.shape)
    print("Batch Labels Shape:", batch_labels.shape)

    print(batch_labels[7])

    # Select the first image in the batch
    image = batch_features[7].squeeze(0)  # Remove the channel dimension (from shape [1, 28, 28] to [28, 28])
    
    # Display the image
    plt.imshow(image, cmap='gray')  # Grayscale image
    plt.axis('off')  # Hide axes for cleaner display
    plt.show()
    
    break

for X, Y in tdataset:
    print(X.shape)
    print(Y.shape) 

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 6, (3,3)), 
            nn.ReLU(),
            nn.Conv2d(6, 6, (3,3)), 
            nn.ReLU(),
            nn.Flatten(), 
            nn.Linear(6*(28-4)*(28-4), 10)  
        )
    def forward(self,x):
        return self.model(x)

In [ ]:
clf = ImageClassifier().to('cuda')
opt = Adam(clf.parameters(), lr=1e-3 )
lossfn = nn.CrossEntropyLoss()

In [ ]:
losses = []
accuracies = []
    
for epoch in range(15): # train for 10 epochs
    correct = 0
    total = 0

    
    for batch in dataset:
        X,y = batch 
        X, y = X.to('cuda'), y.to('cuda') 
        yhat = clf(X) 
        loss = lossfn(yhat, y) 

        # Calculate accuracy
        _, predicted = torch.max(yhat, 1)  # Get the predicted class (argmax over output)
        total += y.size(0)  # Update the total number of samples processed
        correct += (predicted == y).sum().item()  # Update the number of correct predictions

        # Apply backprop 
        opt.zero_grad()
        loss.backward() 
        opt.step() 

    accuracy = (correct / total) * 100  # Compute the accuracy as a percentage
    print(f"Epoch: {epoch} - Loss: {loss.item()} - Accuracy: {accuracy:.2f}%")
    losses.append(loss.item())
    accuracies.append(accuracy)


with open('model_state.pt', 'wb') as f: 
    save(clf.state_dict(), f) 



In [ ]:
print(losses)
plt.plot(np.array(losses))
plt.show()

plt.plot(accuracies)
plt.show()

In [ ]:
with open('model_state.pt', 'rb') as f: 
    state_dict = torch.load(f, weights_only=True)  # Explicitly set weights_only=True
    clf.load_state_dict(state_dict)

for X,Y in tdataset:
    X,Y = X.to('cuda') , Y.to('cuda')
    Yhat = clf(X)
    _, predicted = torch.max(Yhat, 1)
    correct = (predicted == Y).sum().item()
    accuracy = (correct / Y.size(0)) * 100
    print(f'new data accuracy: {accuracy}')

In [ ]:
import os
from scipy.signal import convolve2d

def printAndSave(layerNum, filterNum, channelNum, npfilter, original, convolution):

    folder_name = str(layerNum)
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    
    plt.imshow(npfilter, cmap='gray')
    plt.axis('off')
    plt.savefig(os.path.join(folder_name, f'filter:{layerNum},{filterNum},{channelNum}.png'), bbox_inches='tight', pad_inches=0)
    plt.show()
    plt.imshow(original, cmap='gray')
    plt.axis('off')
    plt.show()
    plt.imshow(convolution, cmap='gray')
    plt.axis('off')
    plt.savefig(os.path.join(folder_name, f'convolution:{layerNum},{filterNum},{channelNum}.png'), bbox_inches='tight', pad_inches=0)
    plt.show()

# Instantiate the model
model = ImageClassifier()

npfilter = np.empty((3, 3))
image = trainData.iloc[100, 1:].values
original = image.reshape(28, 28)

# Iterate through layers and print filters for Conv2d layers
for name, layer in model.model.named_children():  # Access layers inside nn.Sequential
    if isinstance(layer, nn.Conv2d):
        print(f"Layer: {name}")
        print(f"Filters (weights): {layer.weight.data.shape}")
        # indexed as [number of filters, number of channels, filter size, filter size]
        # print(layer.weight.data[2,0,:,:])
        nOfFilters , nOfChannels, _, _ = layer.weight.data.shape
        for F in range(nOfFilters):
            for C in range(nOfChannels):
                npfilter = layer.weight.data[F,C,:,:].detach().cpu().numpy()
                convolution = convolve2d(original, npfilter)
                printAndSave(name,F,C,npfilter,original,convolution)
        print(f"Biases: {layer.bias.data if layer.bias is not None else 'No Bias'}")
        print("-" * 50)

